# NBA - 2050-2017 - EXPLORE HEIGHT OVER TIME

## DATA CLEANSING

## Import packages

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## Read CSV files

In [ ]:
players = pd.read_csv('Players.csv')
seasons = pd.read_csv('Seasons_Stats.csv')
#seasons = seasons.drop(seasons.index[len(seasons) - 1])


### First look at the tables

In [ ]:
# SEASONS TABLE
pd.set_option('display.max_columns', 60)
print(seasons.shape)
#print('seasons table has '+ str(len(seasons))+ ' rows.')
seasons.head(2)

In [ ]:
# PLAYERS TABLE
print(players.shape)
#print('players table has '+ str(len(players))+ ' rows.')
players.head(2)

Remove "Unnamed: 0" column

In [ ]:
seasons.drop('Unnamed: 0',axis=1,inplace=True)
players.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
players.head(2)

### Remove nulls - players table

In [ ]:
players.isnull().sum()
#players.info()

In [ ]:
players = players.dropna(subset=['Player']).reset_index(drop=True)
print(players.shape)
players.isnull().sum()

### Remove nulls - seasons table

In [ ]:
seasons.isnull().sum()

In [ ]:
seasons = seasons.dropna(subset=['Player']).reset_index(drop=True)
print(seasons.shape)
seasons.isnull().sum()

## DATA PREPERATION

In [ ]:
# Create New tables with "Player" as Index
seasons_NI = seasons.set_index('Player',inplace=False) #setting the player name as the seasons index
players_NI = players.set_index('Player',inplace=False) #setting the player name as the players index

In [ ]:
print(players_NI.iloc[:,0:2].drop_duplicates().head())
print(seasons_NI.head())
#players[players.Player == 'Allen Iverson*'].head()
       
seasons_NI.head()

In [ ]:
print(players.shape)
#print(players_NI.iloc[:,0:2])

## Filter Significant players

In [ ]:
#Show data from 1951 and 1952
print(' DATA FROM 1951: \n')
print(seasons_NI[seasons_NI['Year']==1951].loc[:,['G', 'MP']].head()) 
print('\n DATA FROM 1952: \n')
print(seasons_NI[seasons_NI['Year']==1952].loc[:,['G', 'MP']].head()) 

# G= Games, MP = Minutes Played
seasons_NI = seasons_NI[(((seasons_NI['MP']/seasons_NI['G'])>25) | ((seasons_NI['Year']<1952) & (seasons_NI['G']>30)))]

# Join players and seasons tables into "seasons_new" - relevant fields only

In [ ]:
print(players_NI.iloc[:,0:2].head())
print(seasons_NI.iloc[:,0:2].head())

In [ ]:
players_for_join = players_NI.iloc[:,0:2]
seasons_for_join = seasons_NI.iloc[:,0:2]

seasons_new = seasons_for_join.join(players_for_join,on='Player')

In [ ]:
print(seasons_new.head())
print(len(seasons_new))
print(len(seasons_new.drop_duplicates()))
seasons_new.info()

## GROUPING 'POS' INTO 3 POSITIONS

In [ ]:
print (seasons_new.Pos.unique())
unique_count = seasons_new.Pos.nunique()
print(f' There are {unique_count} unique positions out of {len(seasons_new)} records.')

In [ ]:
seasons_new.reset_index(inplace=True)
seasons_new.groupby('Pos')[['Player']].count()

In [ ]:
def changePosition(x):
    #if ("SG" in x) or ("PG" in x):
    if ("G" in x):
        return "Guard"
    #elif ("SF" in x) or ("PF" in x):
    elif ("C" in x):
        return "Center"
    else:
        return "Forward"

In [ ]:
seasons_new['Pos'].head()
seasons_new['Position'] = seasons_new['Pos'].apply(changePosition)
print(seasons_new[seasons_new.Player == 'Allen Iverson*'].head(30))
#seasons.reset_index(inplace=True)
#print(seasons[seasons.Player == 'Allen Iverson*'].head(20))




In [ ]:
seasons_new.drop(['Pos'], axis=1, inplace = True)
seasons_new["is_duplicate"]= seasons_new.duplicated()
#print(seasons_new[seasons_new.Player == 'Isaiah Thomas'].tail(30))
print(seasons_new[seasons_new.Player == 'Allen Iverson*'].head(30))

In [ ]:
seasons_new.drop(['is_duplicate'], axis=1, inplace = True)


In [ ]:
seasons_new = seasons_new.drop_duplicates()
print(seasons_new[seasons_new.Player == 'Allen Iverson*'].head(20))
len(seasons_new)

# GRAPHS

In [ ]:
seasons_new.groupby('Position')[['Player']].count().plot(kind='bar',rot=15, title = '# of Players by Position')

In [ ]:
#print(seasons_new.groupby(['Year','Position'])[['height']].mean())
seasons_new.groupby(['Year'])[['height']].mean().plot(title = 'Avg. Height by Year - Over Time')

In [ ]:
#seasons_new.groupby(['Year'])[['height']].mean().plot(kind='box')
seasons_new[['height']].plot(kind='hist', title = 'Players Height Histogram')
#df_dessert.groupby('num.of.guests')[['dessert', 'not.dessert']].sum().plot(kind='bar')

In [ ]:
seasons_new[(seasons_new['Position']=='Center')].groupby('Year').height.mean().plot(title = 'Avg. Height by Year - Centers')

In [ ]:
#[seasons_new['Position']

fig = plt.figure()
ax = fig.gca()
for n in seasons_new.Position.unique():
    seasons_new[(seasons_new['Position']==n)].groupby('Year').height.mean().plot()

ax.set_title('Avg. Height Change Over Time')
ax.set_ylabel('Height (Centimeters)')
plt.legend(seasons_new.Position.unique(), loc='best')
plt.show()

## ANALYSIS BY DECADE

In [ ]:
def Decade(z):
    if z < 1960:
         return "(1) 50's"
    elif z < 1970:
         return "(2) 60's"
    elif z < 1980:
         return "(3) 70's"
    elif z < 1990:
         return "(4) 80's"
    elif z < 2000:
         return "(5) 90's"
    elif z < 2010:
         return "(6) 2000's"
    else:
         return "(7) 2010's"

In [ ]:
#seasons_new['Year'].apply(Decade)
seasons_new['Decade'] = seasons_new['Year'].apply(Decade)
seasons_new.tail()

In [ ]:
df_pivot=pd.pivot_table(seasons_new,
               index='Decade',
               columns='Position', 
               values='height', 
               aggfunc='mean',
               fill_value=0)
df_pivot.head()

In [ ]:
## 2010's - Guard's game
df_pivot.plot(kind='bar', title='Avg. Height by Decade'); plt.ylim((185,220))

In [ ]:
seasons_new[(seasons_new['Position']=='Guard') ].groupby('Decade').Player.count()
#seasons_new[(seasons_new['Position']=='Guard') & (seasons_new['Decade']=="90's")].head(100)
#seasons_new[(seasons_new['Position']==n) & (seasons_new['Year']>1966)].groupby('Year').height.mean()

In [ ]:
color_map = {'Guard': 'r', 'Forward': 'b', 'Center': 'y'}
ax = plt.subplot(title='Avg. Height by Decade')

x, y = seasons_new.Decade, seasons_new.height
colors = seasons_new.Position.map(color_map)
ax.scatter(x, y, color=colors)
plt.legend(color_map)

## ANECDOTES

In [ ]:
print(' Shortest players in history: \n')
print(seasons_new[((seasons_new['height'])<170)].loc[:,['Player', 'height','Decade']].drop_duplicates())
print('\n Tallest players in history: \n')
print(seasons_new[((seasons_new['height'])>225)].loc[:,['Player', 'height','Decade']].drop_duplicates().sort_values('height',ascending=False))

In [ ]:
print(' Players averaged more than 48 minutes a game: \n')
print(seasons_NI[((seasons_NI['MP']/seasons_NI['G'])>48)].loc[:,['G', 'MP','Year']])
print('\n Players averaged more than 47 minutes a game: \n')
print(seasons_NI[((seasons_NI['MP']/seasons_NI['G'])>47)].loc[:,['G', 'MP','Year']])